## 1. Parameters

Configurable widgets that control **where** data is read from and **how** the Bronze layer behaves on each run.

| Parameter | Default | Purpose |
|---|---|---|
| `catalog` | `workspace` | Unity Catalog where Bronze tables are created |
| `schema` | `ecom_poc` | Schema (database) within the catalog |
| `volume` | `raw_files` | UC Volume holding the source CSV files |
| `batch_id_override` | _(blank = auto)_ | Force a specific batch id instead of the auto-generated fingerprint |
| `full_refresh` | `no` | If `yes`, drop and recreate Bronze tables before loading |
| `run_idempotency_test` | `yes` | Re-run ingestion immediately to prove no duplicate rows are added |

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")
dbutils.widgets.text("volume", "raw_files", "Volume")
dbutils.widgets.text("batch_id_override", "", "Batch id override (leave blank = auto)")
dbutils.widgets.dropdown("full_refresh", "no", ["no", "yes"], "Full refresh (drop + reload Bronze)?")
dbutils.widgets.dropdown("run_idempotency_test", "yes", ["yes", "no"], "Run repeat-load test?")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
VOLUME = dbutils.widgets.get("volume").strip()
BATCH_OVERRIDE = dbutils.widgets.get("batch_id_override").strip()
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "yes"
RUN_IDEMPOTENCY_TEST = dbutils.widgets.get("run_idempotency_test") == "yes"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

## 2. Dataset configuration & Bronze table setup

Defines the source-to-target mapping for each dataset, the audit columns appended to every Bronze row, and the table-creation routine.

* **`DATASETS`** — maps each dataset name to its source CSV file, target Bronze table, and expected source columns.
* **Audit columns** (`_ingested_at`, `_source_file`, `_batch_id`, `_row_hash`, `_row_id`) are added during ingestion to track provenance and enable idempotent reloads.
* **`create_bronze_table()`** — creates (or drops + recreates on full refresh) each Bronze table with all source columns typed as `STRING` plus the audit columns, with column comments.

In [0]:

DATASETS = {
    "orders": {
        "file": "orders.csv",
        "table": f"`{CATALOG}`.`{SCHEMA}`.`bronze_orders`",
        "columns": ["order_id", "customer_id", "order_date", "category",
                    "quantity", "unit_price", "order_status", "country"],
    },
    "transactions": {
        "file": "transactions.csv",
        "table": f"`{CATALOG}`.`{SCHEMA}`.`bronze_transactions`",
        "columns": ["txn_id", "order_id", "payment_method", "amount",
                    "currency", "txn_status", "txn_ts"],
    },
}

AUDIT_COLUMNS = ["_ingested_at", "_source_file", "_batch_id", "_row_hash", "_row_id"]

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

for ds, cfg in DATASETS.items():
    print(f"{ds:13s} {VOLUME_PATH}/{cfg['file']}  ->  {cfg['table']}")
print("Full refresh:", FULL_REFRESH)


In [0]:
AUDIT_COLUMN_COMMENTS = {
    "_ingested_at": "Timestamp when the row was loaded into Bronze",
    "_source_file": "Full path of the source CSV file",
    "_batch_id": "Stable id of the ingested source snapshot (path+size+modified time)",
    "_row_hash": "SHA-256 of all source column values (NULL-safe). Identical rows share the same hash",
    "_row_id": "Unique row identity: batch id + row hash + occurrence number among identical rows",
}


def create_bronze_table(ds):
    cfg = DATASETS[ds]
    table, cols = cfg["table"], cfg["columns"]

    if FULL_REFRESH:
        spark.sql(f"DROP TABLE IF EXISTS {table}")
        print(f"[{ds}] full refresh: dropped {table}")

    source_cols_sql = ",\n    ".join(f"`{c}` STRING" for c in cols)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            {source_cols_sql},
            `_ingested_at` TIMESTAMP,
            `_source_file` STRING,
            `_batch_id`    STRING,
            `_row_hash`    STRING,
            `_row_id`      STRING
        )
        USING DELTA
        COMMENT 'Bronze: raw {cfg["file"]} loaded as STRING. No cleaning. One row per source CSV data row.'
    """)

    # Column comments for the audit columns (best effort - not critical if unsupported)
    for col, comment in AUDIT_COLUMN_COMMENTS.items():
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN `{col}` COMMENT '{comment}'")
        except Exception as e:
            print(f"[{ds}] (note) could not set comment on {col}: {type(e).__name__}")
            break
    print(f"[{ds}] table ready: {table}")

In [0]:
for ds in DATASETS:
    create_bronze_table(ds)

## 3. Ingestion function

The `ingest()` function performs a single-pass, idempotent load for one dataset. Each step is designed to preserve raw fidelity while guaranteeing that re-running with the same source file does not duplicate rows.

1. **Fingerprint the file** — compute a stable batch id from the file path, size, and modification time (or use the override). If the batch already exists in the table, skip loading.
2. **Read as strings** — load every CSV column as `STRING` so that malformed values (bad dates, text in numeric fields, padding) are preserved exactly as in the source.
3. **Verify columns** — assert the CSV header matches the expected column list before any data is written.
4. **Add audit columns** — populate `_ingested_at`, `_source_file`, `_batch_id`, `_row_hash`, and `_row_id` for every row.
5. **Replace that batch** — use `INSERT OVERWRITE` with a `_batch_id` predicate so the same batch can be safely reloaded without affecting other batches.

In [0]:
import hashlib
import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

FIELD_SEP = "\u001f"                 # unit-separator: cannot be confused with normal CSV text
NULL_TOKEN = "\u0000<NULL>\u0000"    # distinguishes a real NULL from the text 'NULL' or ''


def get_batch_id(ds):
    """
    Stable batch id for a given source snapshot.
    Same file (same path, size, modified time) => same id every run => reruns replace, never append.
    """
    cfg = DATASETS[ds]
    path = f"{VOLUME_PATH}/{cfg['file']}"
    info = {f.name: f for f in dbutils.fs.ls(VOLUME_PATH)}[cfg["file"]]
    mtime = getattr(info, "modificationTime", 0) or 0

    if BATCH_OVERRIDE:
        return BATCH_OVERRIDE

    fingerprint = hashlib.sha256(f"{path}|{info.size}|{mtime}".encode()).hexdigest()[:12]
    return f"{ds}_{fingerprint}"


def read_source(ds):
    """Read the CSV with header, NO inference => all columns are strings."""
    path = f"{VOLUME_PATH}/{DATASETS[ds]['file']}"
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .csv(path)
    )


def ingest(ds):
    cfg = DATASETS[ds]
    table, cols = cfg["table"], cfg["columns"]
    path = f"{VOLUME_PATH}/{cfg['file']}"
    batch_id = get_batch_id(ds)

    # --- Guard 1: fail fast if the file does not match the contract ---
    raw = read_source(ds)
    if raw.columns != cols:
        raise ValueError(
            f"[{ds}] column mismatch.\n  expected: {cols}\n  found   : {raw.columns}\n"
            "Fix the source file; Bronze must preserve the contract columns exactly."
        )

    # --- Guard 2: do not mix two different source snapshots in one Bronze table ---
    other_batches = (
        spark.table(table).where(F.col("_batch_id") != batch_id)
        .select("_batch_id").distinct().limit(5).collect()
    )
    if other_batches:
        raise RuntimeError(
            f"[{ds}] {table} already contains other batch(es): {[r['_batch_id'] for r in other_batches]}.\n"
            f"Current batch is '{batch_id}'. If the source file was replaced on purpose, "
            "rerun with full_refresh = yes so snapshots are not mixed."
        )

    # --- Audit columns ---
    hash_input = F.concat_ws(
        FIELD_SEP,
        *[F.coalesce(F.col(c), F.lit(NULL_TOKEN)) for c in cols]
    )

    df = (
        raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(path))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_row_hash", F.sha2(hash_input, 256))
    )

    # Occurrence number among IDENTICAL rows => unique _row_id.
    # The order inside a group does not matter because the rows are identical in content.
    w = Window.partitionBy("_row_hash").orderBy(F.monotonically_increasing_id())
    df = df.withColumn(
        "_row_id",
        F.concat_ws("-", F.col("_batch_id"), F.col("_row_hash"), F.row_number().over(w).cast("string"))
    )

    final_df = df.select(*cols, *AUDIT_COLUMNS)

    # --- Idempotent write: replace ONLY this batch ---
    (
        final_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"_batch_id = '{batch_id}'")
        .saveAsTable(table)
    )

    loaded = spark.table(table).where(F.col("_batch_id") == batch_id).count()
    print(f"[{ds}] batch '{batch_id}' loaded: {loaded:,} rows into {table}")
    return {"dataset": ds, "batch_id": batch_id, "rows_loaded": loaded}

## 4. Run the ingestion for both files

Executes `ingest()` for every dataset in `DATASETS` and stores the results — batch id and row count — for validation in the next section.

In [0]:
ingest_results = {}
for ds in DATASETS:
    ingest_results[ds] = ingest(ds)

print()
print(ingest_results)

## 5. Validation: source vs Bronze parity and schema

Verifies that Bronze row counts match the source files and that all columns landed with the expected `STRING` type.

* **Row-count parity** — the source count is measured by a **fresh, independent read** of the CSV, not by reusing the ingest DataFrame, so the comparison is unbiased.
* **Schema check** — confirms every source column is `STRING` and every audit column has the correct type and comment.
* **Sample inspection** — displays representative rows so you can visually confirm that blanks, text in numeric columns, odd dates, and padded labels were preserved exactly as in the file.

In [0]:

validation_rows = []
all_ok = True

for ds, cfg in DATASETS.items():
    table, cols = cfg["table"], cfg["columns"]
    batch_id = ingest_results[ds]["batch_id"]

    source_count = read_source(ds).count()
    t = spark.table(table)
    bronze_batch = t.where(F.col("_batch_id") == batch_id)
    bronze_count = bronze_batch.count()
    table_total = t.count()

    # schema: every source column must be string; every audit column must exist
    types = {f.name: f.dataType.simpleString() for f in t.schema.fields}
    non_string_source = [c for c in cols if types.get(c) != "string"]
    missing_audit = [c for c in AUDIT_COLUMNS if c not in types]

    # identity: _row_id unique, hash/id never null
    stats = bronze_batch.agg(
        F.countDistinct("_row_id").alias("distinct_row_ids"),
        F.sum(F.col("_row_hash").isNull().cast("int")).alias("null_hash"),
        F.sum(F.col("_row_id").isNull().cast("int")).alias("null_row_id"),
        F.countDistinct("_row_hash").alias("distinct_hashes"),
    ).collect()[0]

    checks = {
        "source rows == bronze rows (this batch)": source_count == bronze_count,
        "table total == batch rows (single snapshot)": table_total == bronze_count,
        "all source columns are STRING": not non_string_source,
        "all audit columns present": not missing_audit,
        "_row_id unique per row": stats["distinct_row_ids"] == bronze_count,
        "no NULL _row_hash / _row_id": (stats["null_hash"] or 0) == 0 and (stats["null_row_id"] or 0) == 0,
    }

    print("=" * 70)
    print(f"{ds}  ->  {table}")
    print(f"  batch id            : {batch_id}")
    print(f"  source rows         : {source_count:,}")
    print(f"  bronze rows (batch) : {bronze_count:,}")
    print(f"  bronze rows (table) : {table_total:,}")
    print(f"  distinct _row_hash  : {stats['distinct_hashes']:,}  (fewer than rows => exact duplicate rows exist)")
    print(f"  distinct _row_id    : {stats['distinct_row_ids']:,}")
    for label, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
        all_ok &= ok

    validation_rows.append((ds, source_count, bronze_count, table_total, source_count == bronze_count))

print("\nOVERALL:", "BRONZE VALID" if all_ok else "BRONZE CHECKS FAILED")
assert all_ok, "Bronze validation failed - see output above."

In [0]:
# Parity table (screenshot-friendly)
display(spark.createDataFrame(
    validation_rows,
    "dataset STRING, source_rows LONG, bronze_rows LONG, table_total_rows LONG, counts_match BOOLEAN"
))

### Schema evidence — all source columns `STRING` + audit columns

`DESCRIBE TABLE` output for each Bronze table confirming the raw-as-string design and the audit column comments.

In [0]:
display(spark.sql(f"DESCRIBE TABLE {DATASETS['orders']['table']}"))

In [0]:
display(spark.sql(f"DESCRIBE TABLE {DATASETS['transactions']['table']}"))

### Sample rows — original values + audit metadata

First 10 rows from each Bronze table. Look for blanks, text in numeric columns, odd dates, and padded labels — all preserved exactly as in the source file.

In [0]:
display(spark.table(DATASETS["orders"]["table"]).limit(10))

In [0]:
display(spark.table(DATASETS["transactions"]["table"]).limit(10))

### Proof that malformed values were NOT cleaned

Queries pulled straight from Bronze for rows with unparseable quantities, invalid dates, or padded status labels. If these return rows, the raw problems were preserved — confirming Bronze does not sanitise or transform source data.

In [0]:
display(spark.sql(f"""
    SELECT order_id, quantity, unit_price, order_date, order_status, _row_id
    FROM {DATASETS['orders']['table']}
    WHERE try_cast(trim(quantity) AS DOUBLE) IS NULL
       OR try_cast(trim(order_date) AS DATE) IS NULL
       OR order_status <> trim(order_status)
    LIMIT 15
"""))

## 6. Repeat-run test (idempotency proof)

Re-runs `ingest()` for every dataset and asserts that:

* Row counts are **unchanged** after the second run.
* The **batch id is the same** (the file fingerprint did not change).
* Only **one distinct batch id** exists in each table (no duplicate batch was appended).

This proves the `INSERT OVERWRITE` + `_batch_id` predicate pattern is idempotent — safe to retry without double-counting.

In [0]:
if RUN_IDEMPOTENCY_TEST:
    before = {ds: spark.table(cfg["table"]).count() for ds, cfg in DATASETS.items()}
    print("Counts BEFORE second run:", before)

    second_results = {ds: ingest(ds) for ds in DATASETS}

    after = {ds: spark.table(cfg["table"]).count() for ds, cfg in DATASETS.items()}
    print("Counts AFTER  second run:", after)

    idem_ok = True
    for ds in DATASETS:
        same_count = before[ds] == after[ds]
        same_batch = ingest_results[ds]["batch_id"] == second_results[ds]["batch_id"]
        distinct_batches = spark.table(DATASETS[ds]["table"]).select("_batch_id").distinct().count()
        ok = same_count and same_batch and distinct_batches == 1
        idem_ok &= ok
        print(f"[{'PASS' if ok else 'FAIL'}] {ds}: same count={same_count}, same batch id={same_batch}, "
              f"distinct batch ids in table={distinct_batches}")

    assert idem_ok, "Idempotency test failed - the same batch was loaded more than once."
    print("\nIDEMPOTENCY: PASS - rerunning did not double-count.")
else:
    print("Skipped (run_idempotency_test = no).")
    

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {DATASETS['orders']['table']}").select(
    "version", "timestamp", "operation", "operationParameters", "operationMetrics"
))

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {DATASETS['transactions']['table']}").select(
    "version", "timestamp", "operation", "operationParameters", "operationMetrics"
))